In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import chess.pgn

In [ ]:
with open("lichess_sample.pgn", "r") as file:
    samplegame = chess.pgn.read_game(file)
    print(samplegame)

[Event "Rated Bullet game"]
[Site "https://lichess.org/AJrdLVya"]
[Date "2026.02.01"]
[Round "-"]
[White "Orangutanagram"]
[Black "bachess_2024"]
[Result "1-0"]
[UTCDate "2026.02.01"]
[UTCTime "00:00:33"]
[WhiteElo "1696"]
[BlackElo "1617"]
[WhiteRatingDiff "+4"]
[BlackRatingDiff "-4"]
[ECO "B12"]
[Opening "Caro-Kann Defense: Advance Variation, Short Variation"]
[TimeControl "60+0"]
[Termination "Time forfeit"]

1. e4 { [%clk 0:01:00] } 1... c6 { [%clk 0:01:00] } 2. d4 { [%clk 0:00:58] } 2... d5 { [%clk 0:01:00] } 3. e5 { [%clk 0:00:58] } 3... Bf5 { [%clk 0:00:59] } 4. Nf3 { [%clk 0:00:57] } 4... e6 { [%clk 0:00:58] } 5. Bd3 { [%clk 0:00:56] } 5... Bxd3 { [%clk 0:00:57] } 6. Qxd3 { [%clk 0:00:56] } 6... c5 { [%clk 0:00:56] } 7. c3 { [%clk 0:00:55] } 7... cxd4 { [%clk 0:00:56] } 8. cxd4 { [%clk 0:00:55] } 8... Nc6 { [%clk 0:00:55] } 9. Nc3 { [%clk 0:00:53] } 9... Nge7 { [%clk 0:00:55] } 10. Ng5 { [%clk 0:00:51] } 10... Ng6 { [%clk 0:00:53] } 11. O-O { [%clk 0:00:49] } 11... Be7 { [%clk 

In [ ]:
# Convert PGN to CSV, first 50,000 games only 
# (feel free to adjust this number, this was chosen to balance performance and analysis).

NUM_GAMES = 50000

import csv
from pathlib import Path
import json
import re

pgn_file = "lichess_sample.pgn"
lichess_stockfish_sample = "lichess_stockfish_sample.csv"
lichess_nofish_sample = "lichess_nofish_sample.csv"

# regex patterns
EVAL_PATTERN = re.compile(r'\[%eval ([^\]]+)\]')
CLK_PATTERN = re.compile(r'\[%clk ([^\]]+)\]')


# Define CSV headers
fieldnames = [
    'event', 'site', 'date', 'white', 'black', 'white_elo', 'black_elo',
    'result', 'eco', 'opening', 'time_control', 'termination', 'num_moves', 'moves'
]

def extract_game_headers(game):
    return {
        'event': game.headers.get('Event', ''),
        'site': game.headers.get('Site', ''),
        'date': game.headers.get('Date', ''),
        'white': game.headers.get('White', ''),
        'black': game.headers.get('Black', ''),
        'white_elo': game.headers.get('WhiteElo', ''),
        'black_elo': game.headers.get('BlackElo', ''),
        'result': game.headers.get('Result', ''),
        'eco': game.headers.get('ECO', ''),
        'opening': game.headers.get('Opening', ''),
        'time_control': game.headers.get('TimeControl', ''),
        'termination': game.headers.get('Termination', ''),
    }

game_count = 0
stockfish_count = 0
nofish_count = 0

# Open CSV files and write headers
with open(lichess_stockfish_sample, "w", newline="", encoding="utf-8") as stockfish_f, \
     open(lichess_nofish_sample, "w", newline="", encoding="utf-8") as nofish_f:
    
    stockfish_writer = csv.DictWriter(stockfish_f, fieldnames=fieldnames)
    nofish_writer = csv.DictWriter(nofish_f, fieldnames=fieldnames)
    stockfish_writer.writeheader()
    nofish_writer.writeheader()
    
    # Process games
    with open(pgn_file, "r", encoding="utf-8") as pgn_f:
        while game_count < NUM_GAMES:
            game = chess.pgn.read_game(pgn_f)
            if game is None:
                break
            
            # Extract moves
            moves_list = list(game.mainline_moves())
            num_moves = len(moves_list)
            
            # Filter out games with 5 or fewer moves (often games that get abandoned early)
            if num_moves <= 5:
                continue
            
            # Convert moves to algebraic notation and check for analysis
            board = game.board()
            node = game
            moves_data = []
            has_analysis = False
            
            for move in moves_list:
                san_move = board.san(move)
                move_info = {'move': san_move}
                
                # Check for analysis
                try:
                    node = node.variation(move)
                    if node.comment:
                        eval_match = EVAL_PATTERN.search(node.comment)
                        if eval_match:
                            move_info['eval'] = eval_match.group(1)
                            has_analysis = True
                        
                        clk_match = CLK_PATTERN.search(node.comment)
                        if clk_match:
                            move_info['clk'] = clk_match.group(1)
                except KeyError:
                    pass
                
                moves_data.append(move_info)
                board.push(move)
            
            # Store moves as JSON
            moves_str = json.dumps(moves_data)
            
            # Create game row
            game_row = extract_game_headers(game)
            game_row.update({
                'num_moves': num_moves,
                'moves': moves_str
            })
            
            # Write to appropriate CSV file
            writers = {
                True: (stockfish_writer, 'stockfish_count'),
                False: (nofish_writer, 'nofish_count')
            }
            
            writer, count_var_name = writers[has_analysis]
            writer.writerow(game_row)
            
            # Update counter
            if has_analysis:
                stockfish_count += 1
            else:
                nofish_count += 1
            
            game_count += 1
            
            if game_count % 5000 == 0:
                print(f"Processed {game_count:,} games...")

print(f"Total games processed: {game_count:,}")
print(f"Games with stockfish analysis: {stockfish_count}")
print(f"Games without stockfish analysis: {nofish_count}")
print(f"CSV files created:")
print(f"  - {lichess_stockfish_sample}")
print(f"  - {lichess_nofish_sample}")

Processed 5,000 games...
Processed 10,000 games...
Processed 15,000 games...
Processed 20,000 games...
Processed 25,000 games...
Processed 30,000 games...
Processed 35,000 games...
Processed 40,000 games...
Processed 45,000 games...
Processed 50,000 games...
Total games processed: 50,000
Games with stockfish analysis: 5445
Games without stockfish analysis: 44555
CSV files created:
  - lichess_stockfish_sample.csv
  - lichess_nofish_sample.csv
